# LogisticRegression & Performance Metrics

This stage involves 3 components:
- Train & Test Model
- Calculate metrics to evaluate model performance
- SHAP analysis to maintain explainability

All three components will be done in this Colab Notebook - **06_PPA_logisticregression_shap_metrics.ipynb**

## Training & Testing LogisticRegression
- Input (X) is the acoustic and linguistic features and output (Y) is diagnosis
- **ppa_master_data_frame_train.csv** was used for **training** the model
- **ppa_master_data_frame_test.csv** was used for **testing** the model

## Calculating Performance Metrics
- **Learning Curve** - Reveals how model performance changes with more data and checks for underfitting/overfitting
- **Classification Report** - Includes Accuracy, Precision, Recall, F1 Score
- **Multi-Class AUC-ROC Score** - Measures model ability to classify between (PPA) variants
- **Confusion Matrix** - Compares actual diagnosis aganist predicted classifications
- **Multi-Class AUC-ROC Plot** - Multi-line graph that plots TPR vs. FPR for each individual group
- **PR-AUC** - Measures precision and recall across different checkpoints (better for imbalanced data)

## SHAP (SHapley Additive exPlanations) analysis
-  Calculates feature importance and its contribution to model's predictions
- Helpful in revealing how the model reached its conclusion
- **Global Bar Chart** - Ranks features based on importance and impact across all patient groups
- **Beeswarm Plot** - Shows how high or low values influence model's classification for each individual group

In [ ]:
# Install required libraries (comment out after installing once)
# !pip install pandas==2.2.2  # For handling CSV files
# !pip install numpy==2.0.2 # For math calculations
# !pip install shap==0.52.0  # Explainable AI
# !pip install scikit-learn==1.6.0 # Has ML model
# !pip install matplotlib==3.10.0 # For data visualization
# !pip install seaborn==0.13.2 # For data visualization

# Import required libraries
import os # File Path handling
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
from sklearn.linear_model import LogisticRegression # Loads Logistic Regression model
from sklearn.pipeline import Pipeline # Prevents data leakage
from sklearn.preprocessing import StandardScaler, label_binarize # StandardScaler for Logistic Regression & Multi-class ROC
from sklearn.model_selection import StratifiedKFold, cross_validate, learning_curve # Stratified K-Fold cross validation and learning curves
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, precision_recall_fscore_support, balanced_accuracy_score, precision_recall_curve, auc, average_precision_score # Metrics calculation
from matplotlib.colors import ListedColormap # SHAP globar bar color palette

# Set font size guidelines
plt.rcParams.update({
    'font.size': 12,            # Base text is 12
    'axes.labelsize': 14,       # X and Y axis labels are 14
    'axes.labelweight': 'bold', # X and Y axis labels are bold
    'xtick.labelsize': 12,      # X-axis numbers - 12
    'ytick.labelsize': 12       # Y-axis numbers - 12
})

# 1. Connect Drive and Colab
from google.colab import drive
drive.mount('/content/drive')

# 2. Load Train and Test CSV files
# Establish File Path
data_dir = "/content/drive/MyDrive/Research Spike (Set 3+7)/Research Internships/YRI Fellowship - Vansika Priya Garapati/YRI Research Materials/Results/csv_files/v3 train and test"

# Open both CSV files as DataFrames
train_df = pd.read_csv(os.path.join(data_dir, "ppa_master_data_frame_train.csv")) # Loads training data
test_df = pd.read_csv(os.path.join(data_dir, "ppa_master_data_frame_test.csv")) # Loads test data

# 3. Separate features (X) and target (y)
non_feature_cols = ['participant_id', 'diagnosis'] # Lists informational column names
feature_cols = [c for c in train_df.columns if c not in non_feature_cols] # Gathers all numeric columns

X_train = train_df[feature_cols] # Extracts only feature data
y_train = train_df['diagnosis'] # Extracts only diagnosis

X_test = test_df[feature_cols] # Extracts only feature data
y_test = test_df['diagnosis'] # Extracts only diagnosis

# 4. Set up Stratified K-Fold Cross-Validation (Ensures class proportions are balanced)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 5. Train Logistic Regression Classifier with Pipeline and Class Weighting
model_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42))
])

# Stratified Cross-Validation on Training Data
cv_results = cross_validate(model_pipeline, X_train, y_train, cv=skf, scoring=['balanced_accuracy', 'f1_macro']) # Runs 5-fold stratified cross-validation
print("Stratified 5-Fold CV Balanced Accuracy:", np.mean(cv_results['test_balanced_accuracy'])) # Displays mean cross validation accuracy
print("Stratified 5-Fold CV Macro F1-Score:", np.mean(cv_results['test_f1_macro'])) # Displays mean cross validation macro F1 score

model_pipeline.fit(X_train, y_train) # Final model on full training dataset
print("Model training complete!")

# 6. Make predictions on test dataset
y_pred = model_pipeline.predict(X_test) # Predicted diagnoses
y_probs = model_pipeline.predict_proba(X_test) # Calculates confidence probability
classes = np.unique(y_test) # List of diagnosis names in train data
y_test_bin = label_binarize(y_test, classes=classes) # Converts diagnosis categories into binary numbers (0,1) for ROC graphing

print("Model testing complete!")

# Figures & Metrics
# Establish new file path
output_dir = "/content/drive/MyDrive/Research Spike (Set 3+7)/Research Internships/YRI Fellowship - Vansika Priya Garapati/YRI Research Materials/Results/figures_tables/v8 figures and tables"
os.makedirs(output_dir, exist_ok=True)

# 1. Learning Curve using StratifiedKFold
train_sizes, train_scores, test_scores = learning_curve(  # train_sizes = patient checkpoints, train_scores = train accuracy, test_scores = cv validation accuracy
    Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42))
    ]), # Model for generating learning curve
    X_train, y_train, cv=skf, scoring='balanced_accuracy', # Uses StratifiedKFold cross-validation with balanced accuracy scoring
    train_sizes=np.linspace(0.2, 1.0, 5), random_state=42 # 5 checkpoints (20% each time)
)
plt.figure(figsize=(8, 6)) # Creates blank canvas set to 8 in wide, 6 in high
plt.plot(train_sizes, np.mean(train_scores, axis=1), 'o-', color="blue", label="Training Accuracy") # Plots training accuracy curve
plt.plot(train_sizes, np.mean(test_scores, axis=1), 'o-', color="orange", label="Testing Accuracy") # Plots validation accuracy curve
plt.xlabel("Number of Data Samples Used") # X-axis label
plt.ylabel("Balanced Accuracy Score") # Y-axis label
plt.legend(loc="lower right") # Legend position
plt.grid(True) # Adds grid lines
plt.tight_layout() # Adjusts layout padding
plt.savefig(os.path.join(output_dir, "learning_curve.tiff"), format='tiff', dpi=300, bbox_inches='tight') # Saves figure to Google Drive
plt.show() # Displays graph


# 2. Classification Report & Imbalance Metrics
bal_acc = balanced_accuracy_score(y_test, y_pred) # Calculates Balanced Accuracy Score on test dataset
print(f"Test Set Balanced Accuracy Score: {bal_acc:.4f}") # Displays test balanced accuracy
print("Classification Report:")
print(classification_report(y_test, y_pred, zero_division=0)) # Calculates and displays Precision, Recall, F1-Score, and Accuracy

# 3. Multi-Class AUC-ROC Score
macro_auc = roc_auc_score(y_test, y_probs, multi_class='ovr', average='macro') # Macro average AUC score
micro_auc = roc_auc_score(y_test_bin, y_probs, average='micro') # Micro average AUC score
print(f"Multi-Class AUC-ROC Score (Macro): {macro_auc:.4f}") # Displays macro AUC score
print(f"Multi-Class AUC-ROC Score (Micro): {micro_auc:.4f}\n") # Displays micro AUC score

# Save Scores/Metrics
report_text = f"Stratified 5-Fold CV Mean Balanced Accuracy: {np.mean(cv_results['test_balanced_accuracy']):.4f}\n"
report_text += f"Stratified 5-Fold CV Mean Macro F1-Score: {np.mean(cv_results['test_f1_macro']):.4f}\n"
report_text += f"Test Set Balanced Accuracy Score: {bal_acc:.4f}\n\n"
report_text += "Classification Report\n"
report_text += classification_report(y_test, y_pred, zero_division=0)
report_text += f"\nAUC-ROC Scores\n"
report_text += f"Macro Average AUC: {macro_auc:.4f}\n"
report_text += f"Micro Average AUC: {micro_auc:.4f}\n"
print(report_text)
# Save to Drive as a text file
with open(os.path.join(output_dir, "classification_report_output.txt"), "w") as f:
    f.write(report_text) # Writes summary metrics to text file

# 4. Confusion Matrix
plt.figure(figsize=(8, 6)) # Creates blank canvas set to 8 in wide, 6 in high
cm = confusion_matrix(y_test, y_pred, labels=classes) # Calculates number of correct and incorrect predictions per variant
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes, linewidths=1.5,linecolor='white') # Draws matrix with integer counts shaded in blue
plt.xlabel('Predicted Diagnosis') # X-axis label
plt.ylabel('True Diagnosis') # Y-axis label
plt.tight_layout() # Adjusts layout so text doesn't clip
plt.savefig(os.path.join(output_dir, "confusion_matrix.tiff"), format='tiff', dpi=300, bbox_inches='tight') # Saves figure to Google Drive
plt.show() # Displays graph

# 5. Multi-Class AUC-ROC Plot
plt.figure(figsize=(8, 6)) # Creates blank canvas set to 8 in wide, 6 in high
for i, col in enumerate(classes): # Iterates through each diagnosis class
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_probs[:, i]) # Calculates False Positive Rate (FPR) and True Positive Rate (TPR)
    class_auc = roc_auc_score(y_test_bin[:, i], y_probs[:, i]) # Calculates individual AUC score for that class
    plt.plot(fpr, tpr, label=f'{col} (AUC = {class_auc:.2f})') # Plots ROC line and adds class AUC to legend

plt.plot([0, 1], [0, 1], 'k--', linestyle='--') # Draws black dashed line for random guessing (AUC = 0.50)
plt.xlabel('False Positive Rate') # X-axis label
plt.ylabel('True Positive Rate') # Y-axis label
plt.legend(loc='lower right') # Legend position
plt.grid(True) # Grid lines
plt.tight_layout() # Layout adjustments
plt.savefig(os.path.join(output_dir, "roc_curves.tiff"), format='tiff', dpi=300, bbox_inches='tight') # Saves ROC figure to Google Drive
plt.show() # Displays graph

# 6. Model Performance Summary Table
precision, recall, f1, _ = precision_recall_fscore_support(y_test, y_pred, labels=classes, zero_division=0) # Extracts precision, recall, f1
auc_list = [roc_auc_score(y_test_bin[:, i], y_probs[:, i]) for i in range(len(classes))] # Calculates ROC-AUC scores for each group

# Calculate PR-AUC for each group (Better for rare groups)
pr_auc_list = [] # Initializes PR-AUC storage
for i in range(len(classes)): # Goes over groups
    p_curve, r_curve, _ = precision_recall_curve(y_test_bin[:, i], y_probs[:, i]) # Calculates precision recall curve points
    pr_auc_list.append(average_precision_score(y_test_bin[:, i], y_probs[:, i])) # Calculates PR-AUC using binary (0/1) and probabilities


metrics_df = pd.DataFrame({ # 4 means round to 4 decimal places
    'Diagnosis Variant': classes,
    'Precision': np.round(precision, 4),
    'Recall': np.round(recall, 4),
    'F1-Score': np.round(f1, 4),
    'AUC-ROC': np.round(auc_list, 4),
    'PR-AUC': np.round(pr_auc_list, 4)
}) # Combines metric values into a table

display(metrics_df) # Displays table
metrics_df.to_csv(os.path.join(output_dir, "model_performance_table.csv"), index=False) # Save CSV file in Drive
print("\nSuccess! Figures and table exported to Google Drive.")

# SHAP Analysis
print("Calculating SHAP values for explainability...")
scaler = model_pipeline.named_steps['scaler']
classifier = model_pipeline.named_steps['classifier']

X_train_scaled = pd.DataFrame(scaler.transform(X_train), columns=X_train.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

explainer = shap.LinearExplainer(classifier, X_train_scaled) # SHAP LinearExplainer for Logistic Regression
shap_values = explainer.shap_values(X_test_scaled) # Calculates SHAP contribution scores

# 1. Global Bar Chart
cb_palette = ListedColormap(['#ff7f0e', '#1f77b4', '#9467bd', '#17becf']) # Orange, Blue, Purple, Teal
plt.figure(figsize=(10, 6)) # Creates blank canvas that is 10 in wide, 6 in high
shap.summary_plot(shap_values, X_test_scaled, plot_type="bar", class_names=classes, color=cb_palette, show=False) # Plots ranked mean absolute SHAP feature importance
plt.xlabel("SHAP Value")
plt.ylabel("Linguistic and Acoustic Features")
plt.tight_layout() # Adjusts spacing
plt.savefig(os.path.join(output_dir, "shap_global_bar.tiff"), format='tiff', dpi=300, bbox_inches='tight') # Saves SHAP plot to Drive
plt.show() # Displays plot

# 2. Beeswarm Plots for EACH group
for i, class_name in enumerate(classes): # Goes over each disease variant
    plt.figure(figsize=(10, 6)) # Blank Canvas that is 10 in wide, 6 in high

    class_shap = shap_values[i] if isinstance(shap_values, list) else (shap_values[:, :, i] if len(shap_values.shape) == 3 else shap_values) # Extract class i SHAP values (handles both list and 3D arrays)

    shap.summary_plot(class_shap, X_test_scaled, show=False) # Plots individual SHAP beeswarm scatter plot
    plt.xlabel("SHAP Value")
    plt.ylabel("Linguistic and Acoustic Features")
    plt.tight_layout() # Layout adjustment
    clean_class_name = class_name.replace(" ", "_").lower() # Cleans group name for saving file
    plt.savefig(os.path.join(output_dir, f"shap_beeswarm_{clean_class_name}.tiff"), format='tiff', dpi=300, bbox_inches='tight') # Saves plot per group
    plt.show() # Displays plot

print("All figures and tables have been saved to Drive!") # Final completion print

### Result-Ready Figures & Tables
Changes include:-


1.   Combining **classification_report_output.txt** with **model_performance_table.csv** to generate a master data table of performance metrics.
2.   Dividing master **confusion_matrix.png** into 4 distinct figures for each group (PPA variant or Control)


This is to ensure figures/tables are organized and easy to interpret (clarity).


In [ ]:
# CHANGE 1: Master Data Table
# Import libraries
import os
import pandas as pd

# Define output folder path
output_dir = "/content/drive/MyDrive/Research Spike (Set 3+7)/Research Internships/YRI Fellowship - Vansika Priya Garapati/YRI Research Materials/Results/figures_tables/v6 figures and tables"
os.makedirs(output_dir, exist_ok=True)

# Metrics Table
master_data = {
    'Diagnosis Variant': ['control', 'lvPPA', 'nfvPPA', 'svPPA', 'Macro Average'],
    'Precision': ['1.00', '0.50', '0.50', '0.00', '0.50'],
    'Recall': ['1.00', '0.67', '0.67', '0.00', '0.58'],
    'F1-Score': ['1.00', '0.57', '0.57', '0.00', '0.54'],
    'PR-AUC': ['1.00', '0.55', '0.36', '0.53', '0.61']
}

# Convert to df
master_df = pd.DataFrame(master_data)

# Save and Display
display(master_df)
master_df.to_csv(os.path.join(output_dir, "master_model_performance_summary.csv"))
print("Saved Master CSV to Drive!")

In [ ]:
# CHANGE 2: Confusion Matrix for each group
# Import libraries
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Set font size guidelines
plt.rcParams.update({
    'font.size': 12,            # Base text is 12
    'axes.labelsize': 14,       # X and Y axis labels are 14
    'axes.labelweight': 'bold', # X and Y axis labels are bold
    'xtick.labelsize': 12,      # X-axis numbers - 12
    'ytick.labelsize': 12       # Y-axis numbers - 12
})

# Goes through each diagnosis name in the classes list
for class_name in classes:

    # Blank canavas that is 6 inches wide and 5 inches tall
    plt.figure(figsize=(6, 5))

    # Converts true diagnoses into binary format (1 if it matches, 0 if it doesn't)
    y_test_binary = (y_test == class_name).astype(int)

   # Converts predicted diagnoses into binary format (1 if it matches, 0 if it doesn't)
    y_pred_binary = (y_pred == class_name).astype(int)

    # Generate confusion matrix
    cm_binary = confusion_matrix(y_test_binary, y_pred_binary)
    sns.heatmap(cm_binary, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=['Other', class_name],
                yticklabels=['Other', class_name],
                linewidths=1.5,
                linecolor='white')


    # X-Axis
    plt.xlabel('Predicted Diagnosis')

    # Y-Axis
    plt.ylabel('True Diagnosis')

    # Adjust layout
    plt.tight_layout()

    # Convert spaces to underscores and letters to lowercase for better file naming
    group_name = class_name.replace(" ", "_").lower()

    # File Path
    file_path = os.path.join(output_dir, f"confusion_matrix_{group_name}.tiff")

    # Save to Drive
    plt.savefig(file_path, format='tiff', dpi=300, bbox_inches='tight')

    # Display and close the figures
    plt.show()
    plt.close()

    # Print a success message confirming the specific filename that was saved
    print(f"Saved: confusion_matrix_{group_name}.tiff")